#Customer processing child company

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, coalesce, lit
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
print (bronze_schema_child, silver_schema_child,  gold_schema)

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")
dbutils.widgets.text("company", "child", "Company")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
company = dbutils.widgets.get("company")

base_path = f's3://bucket-storage-company-parents-child/{company}/{data_source}/*.csv'
print(base_path)

##Bronze

In [0]:
df= (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{bronze_schema_child}.{data_source}")

##Silver

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema_child}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

**Transformations**


In [0]:
#- 1: Drop Duplicates
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
# - 2: Trim spaces in customer name
# check those values
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

## remove those trim values

df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
# - 3: Data Quality Fix: Correcting City Typos
df_silver.select('city').distinct().show()

In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
# Sanity check
df_silver.select('city').distinct().show()

- 4: Fix Title-Casing Issue

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

- 5: Handling missing cities

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)


In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:

# - 6: Convert customer_id to string
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

 ### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
# Standardizing Customer Attributes to Match Parent Company Data Model

df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws(" - ", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema_child}.{data_source}")

###Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema_child}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.child_dim_{data_source}")